# Lab 01 — Data Preparation, Model Training & Evaluation for Cybersecurity AI
**Dataset:** UNSW-NB15 (`UNSW_NB15_training-set.csv` / `UNSW_NB15_testing-set.csv`)
**Objective:** full ML pipeline — Preparation → Training → Evaluation — with no data leakage, imbalance handling, multi-class vs binary comparison.
**Models:** Logistic Regression (baseline) + Random Forest — all inside `sklearn Pipeline`.
**Reproducibility:** `RANDOM_STATE = 42` everywhere. Transformers fitted **only on training data**.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             classification_report, confusion_matrix, ConfusionMatrixDisplay,
                             roc_auc_score, roc_curve)
from sklearn.preprocessing import label_binarize

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 120
print("sklearn/pandas/numpy ready. random_state =", RANDOM_STATE)


## 1. Load the two provided CSV files
> Lab requirement: load train + test separately, never mix. Test set stays untouched until final evaluation.


In [ ]:
TRAIN_PATH = "UNSW_NB15_training-set.csv"
TEST_PATH  = "UNSW_NB15_testing-set.csv"

train_df = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)

print(f"Train shape: {train_df.shape}  (expected ~175341 rows)")
print(f"Test  shape: {test_df.shape}  (expected ~82332 rows)")
assert list(train_df.columns) == list(test_df.columns), "Train/test columns differ!"
print("\nColumns (%d):" % len(train_df.columns))
print(list(train_df.columns))
train_df.head(3)


## 2. Exploratory Data Analysis — shape, dtypes, missing values, class distribution


In [ ]:
print("--- Dtypes ---")
print(train_df.dtypes)
print("\n--- Missing values (train) ---")
print(train_df.isna().sum()[train_df.isna().sum() > 0])
print("Total NaN train:", int(train_df.isna().sum().sum()), "| test:", int(test_df.isna().sum().sum()))

# Inf check (numeric only) — required by lab even if count is 0
num_cols_all = train_df.select_dtypes(include=[np.number]).columns.tolist()
n_inf = int(np.isinf(train_df[num_cols_all].values).sum())
print("Total Inf train:", n_inf)

print("\n--- label distribution (0=Normal, 1=Attack) ---")
print(train_df["label"].value_counts())
print(train_df["label"].value_counts(normalize=True).round(4))
print("\n--- attack_cat distribution (10 classes) ---")
print(train_df["attack_cat"].value_counts())
print("\nTest attack_cat:")
print(test_df["attack_cat"].value_counts())


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Label distribution
sns.countplot(x="label", data=train_df, ax=axes[0], palette="Blues_r")
axes[0].set_title("Train: Normal (0) vs Attack (1)")
axes[0].set_xlabel("label")
for c in axes[0].containers:
    axes[0].bar_label(c)

# attack_cat distribution (log scale to show rare Worms/Shellcode)
order = train_df["attack_cat"].value_counts().index
sns.countplot(y="attack_cat", data=train_df, order=order, ax=axes[1], palette="viridis")
axes[1].set_xscale("log")
axes[1].set_title("Train: attack_cat (log scale — note Worms=130)")
plt.tight_layout()
plt.show()

# Categorical cardinality: proto / service / state
print("proto unique:", train_df["proto"].nunique(), "| top5:", train_df["proto"].value_counts().head(5).to_dict())
print("service unique:", train_df["service"].nunique(), "| values:", train_df["service"].value_counts().to_dict())
print("state unique:", train_df["state"].nunique(), "| values:", train_df["state"].value_counts().to_dict())


**Observations (for report):**
- Train 68% Attack / 32% Normal — moderately imbalanced binary, but **severely imbalanced multi-class** (Generic 40000 vs Worms 130, ratio ~300:1).
- `service='-'` (53%) means no service — keep as a valid category, do NOT drop.
- `proto` has 133 values but tcp/udp/unas dominate; rare protos will be grouped by `min_frequency` to limit dimensionality.
- No NaN/Inf in this release, but the pipeline still handles them (robustness + lab requirement).


## 3. Cleaning + Preprocessing design (leakage-proof) + validation split
**Rules applied:**
1. Drop `id` (identifier, would leak / overfit).
2. Strip/lowercase categoricals, replace Inf with NaN then impute.
3. `ColumnTransformer`: numeric → median impute + `StandardScaler`; categorical (`proto,service,state`) → impute + `OneHotEncoder(handle_unknown='ignore', min_frequency=...)`.
4. **Fit transformers ONLY on training data** via `Pipeline`. Never `fit` on val/test.
5. Stratified validation split from train (20%), stratified on `attack_cat` so rare Worms stay represented. Same indices reused for Exp1/Exp2 for fair comparison.


In [ ]:
# --- 3.1 Clean: drop id, define feature lists ---
ID_COL = "id"
CAT_COLS = ["proto", "service", "state"]

def clean_frame(df):
    df = df.copy()
    # normalize categorical strings: strip + lowercase (keeps '-' as its own category)
    for c in CAT_COLS:
        df[c] = df[c].astype(str).str.strip().str.lower()
        df[c] = df[c].replace({"": "missing", "nan": "missing", "-": "missing"})
    # replace Inf with NaN so imputer can handle it (0 Inf here, but methodologically required)
    df = df.replace([np.inf, -np.inf], np.nan)
    return df

train_df = clean_frame(train_df)
test_df  = clean_frame(test_df)

drop_cols = [ID_COL]
FEATURES = [c for c in train_df.columns if c not in drop_cols + ["attack_cat", "label"]]
NUM_COLS = [c for c in FEATURES if c not in CAT_COLS]
print(f"Numeric: {len(NUM_COLS)} | Categorical: {CAT_COLS} | Total features: {len(FEATURES)}")
print(NUM_COLS)


In [ ]:
# --- 3.2 Leakage-proof preprocessor + stratified validation split ---
preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())           # lab: StandardScaler; fit only on train via Pipeline
    ]), NUM_COLS),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=100, sparse_output=True))
    ]), CAT_COLS),
], verbose_feature_names_out=False)

# Stratified split of the TRAIN file -> inner train / validation (test file untouched!)
# Stratify on attack_cat (finest granularity) so Exp1/Exp2/Exp3 splits stay consistent.
X_full = train_df[FEATURES]
y_multi_full = train_df["attack_cat"]
y_bin_full   = train_df["label"]

X_tr, X_val, y_multi_tr, y_multi_val, y_bin_tr, y_bin_val = train_test_split(
    X_full, y_multi_full, y_bin_full,
    test_size=0.20, random_state=RANDOM_STATE, stratify=y_multi_full)

print(f"Inner-train: {X_tr.shape} | Validation: {X_val.shape} | Held-out test: {test_df.shape}")
print("\nValidation attack_cat counts (Worms must be > 0):")
print(y_multi_val.value_counts().tail(4))

# Class imbalance summary (train portion)
print("\n--- Imbalance analysis (inner-train) ---")
print("Binary:", y_bin_tr.value_counts().to_dict())
imb = y_multi_tr.value_counts()
print("Multi-class max/min ratio:", round(imb.max() / imb.min(), 1), ": 1  (Generic vs Worms)")


## 4. Shared evaluation helpers (confusion matrix, macro/weighted, ROC-AUC)
- Binary ROC-AUC: standard `roc_auc_score`.
- Multi-class ROC-AUC: One-vs-Rest macro (`label_binarize` + `roc_auc_score(multi_class='ovr', average='macro')`).


In [ ]:
def scores_binary(y_true, y_pred, y_proba=None):
    d = {"accuracy": accuracy_score(y_true, y_pred),
         "precision": precision_score(y_true, y_pred, zero_division=0),
         "recall": recall_score(y_true, y_pred, zero_division=0),
         "f1": f1_score(y_true, y_pred, zero_division=0)}
    d["roc_auc"] = float(roc_auc_score(y_true, y_proba)) if y_proba is not None else np.nan
    return d

def scores_multi(y_true, y_pred, y_proba=None, labels=None):
    d = {"accuracy": accuracy_score(y_true, y_pred),
         "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
         "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
         "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
         "precision_weighted": precision_score(y_true, y_pred, average="weighted", zero_division=0),
         "recall_weighted": recall_score(y_true, y_pred, average="weighted", zero_division=0),
         "f1_weighted": f1_score(y_true, y_pred, average="weighted", zero_division=0)}
    try:
        if y_proba is not None and labels is not None:
            Yb = label_binarize(y_true, classes=list(labels))
            d["roc_auc_ovr_macro"] = float(roc_auc_score(Yb, y_proba, multi_class="ovr", average="macro"))
        else:
            d["roc_auc_ovr_macro"] = np.nan
    except Exception:
        d["roc_auc_ovr_macro"] = np.nan
    return d

def plot_cm(y_true, y_pred, title, figsize=(6, 5)):
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=figsize)
    ConfusionMatrixDisplay(cm).plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(title)
    plt.tight_layout(); plt.show()
    return cm


## 5. Experiment 1 — Multi-class classification (`attack_cat`, 10 classes)
Models: Logistic Regression (baseline) + Random Forest. Simple hyperparameter search **on validation**, then final eval on held-out test.


In [ ]:
# --- Exp1: simple hyperparameter search on VALIDATION (no test leakage) ---
from sklearn.base import clone

def make_pipe(model):
    return Pipeline([("prep", clone(preprocessor)), ("clf", model)])

exp1_configs = {
    "LogReg C=0.5":  LogisticRegression(C=0.5, max_iter=500, random_state=RANDOM_STATE),
    "LogReg C=1.0":  LogisticRegression(C=1.0, max_iter=500, random_state=RANDOM_STATE),
    "LogReg C=5.0":  LogisticRegression(C=5.0, max_iter=500, random_state=RANDOM_STATE),
    "RF d=None/100": RandomForestClassifier(n_estimators=100, max_depth=None, min_samples_split=2, n_jobs=-1, random_state=RANDOM_STATE),
    "RF d=20/200":   RandomForestClassifier(n_estimators=200, max_depth=20, min_samples_split=2, n_jobs=-1, random_state=RANDOM_STATE),
    "RF d=15/100":   RandomForestClassifier(n_estimators=100, max_depth=15, min_samples_split=5, n_jobs=-1, random_state=RANDOM_STATE),
}

exp1_val_rows = []
exp1_pipes = {}
for name, clf in exp1_configs.items():
    pipe = make_pipe(clf)
    pipe.fit(X_tr, y_multi_tr)                       # fit prep ONLY on inner-train
    tr_f1 = f1_score(y_multi_tr, pipe.predict(X_tr), average="weighted")
    va_f1 = f1_score(y_multi_val, pipe.predict(X_val), average="weighted")
    exp1_val_rows.append({"model": name, "train_f1_weighted": round(tr_f1, 4), "val_f1_weighted": round(va_f1, 4),
                          "gap(train-val)": round(tr_f1 - va_f1, 4)})
    exp1_pipes[name] = pipe
    print(f"{name:14s} train_F1w={tr_f1:.4f}  val_F1w={va_f1:.4f}  gap={tr_f1-va_f1:+.4f}")

exp1_search = pd.DataFrame(exp1_val_rows).sort_values("val_f1_weighted", ascending=False)
print("\nBest on validation (highest val F1-weighted):")
print(exp1_search.to_string(index=False))


In [ ]:
# --- Exp1: pick best LogReg + best RF, evaluate on HELD-OUT TEST ---
best_lr_name = exp1_search[exp1_search["model"].str.startswith("LogReg")].iloc[0]["model"]
best_rf_name = exp1_search[exp1_search["model"].str.startswith("RF")].iloc[0]["model"]
print("Best LogReg:", best_lr_name, "| Best RF:", best_rf_name)

X_test = test_df[FEATURES]
y_multi_test = test_df["attack_cat"]
LABELS_MULTI = sorted(y_multi_full.unique())

exp1_test_rows = []
for name in [best_lr_name, best_rf_name]:
    pipe = exp1_pipes[name]
    yp = pipe.predict(X_test)
    try:
        ypr = pipe.predict_proba(X_test)
    except Exception:
        ypr = None
    s = scores_multi(y_multi_test, yp, ypr, LABELS_MULTI)
    s["model"] = name
    exp1_test_rows.append(s)
    print(f"\n===== {name} — TEST classification_report (Exp1 multi-class) =====")
    print(classification_report(y_multi_test, yp, zero_division=0))
    plot_cm(y_multi_test, yp, f"Exp1 TEST CM — {name}", figsize=(8, 6))

exp1_test = pd.DataFrame(exp1_test_rows).set_index("model")
print("\nExp1 TEST summary:")
print(exp1_test.round(4).to_string())


**Exp1 reading guide (overfitting / underfitting):**
- Large positive `gap(train-val)` (e.g. RF depth=None) = overfitting; small gap + low scores = underfitting (LogReg on 10 classes).
- Expect RF ≫ LogReg on weighted-F1, but **macro-F1 stays low** for both (rare Worms/Shellcode/Backdoor drag the mean) — key point for the report.


## 6. Experiment 2 — Binary classification (`label`: Normal vs Attack)
Same pipelines retrained with binary target. Compare against Exp1.


In [ ]:
exp2_configs = {
    "LogReg C=1.0": LogisticRegression(C=1.0, max_iter=500, random_state=RANDOM_STATE),
    "LogReg C=5.0": LogisticRegression(C=5.0, max_iter=500, random_state=RANDOM_STATE),
    "RF d=None/100": RandomForestClassifier(n_estimators=100, max_depth=None, n_jobs=-1, random_state=RANDOM_STATE),
    "RF d=20/200":   RandomForestClassifier(n_estimators=200, max_depth=20, n_jobs=-1, random_state=RANDOM_STATE),
}

y_bin_test = test_df["label"]
exp2_val_rows, exp2_pipes = [], {}
for name, clf in exp2_configs.items():
    pipe = make_pipe(clf)
    pipe.fit(X_tr, y_bin_tr)
    tr_f1 = f1_score(y_bin_tr, pipe.predict(X_tr))
    va_f1 = f1_score(y_bin_val, pipe.predict(X_val))
    exp2_val_rows.append({"model": name, "train_f1": round(tr_f1, 4), "val_f1": round(va_f1, 4), "gap": round(tr_f1-va_f1, 4)})
    exp2_pipes[name] = pipe
    print(f"{name:14s} train_F1={tr_f1:.4f} val_F1={va_f1:.4f} gap={tr_f1-va_f1:+.4f}")

exp2_search = pd.DataFrame(exp2_val_rows).sort_values("val_f1", ascending=False)
print(exp2_search.to_string(index=False))

best_lr2 = exp2_search[exp2_search["model"].str.startswith("LogReg")].iloc[0]["model"]
best_rf2 = exp2_search[exp2_search["model"].str.startswith("RF")].iloc[0]["model"]

exp2_test_rows = []
for name in [best_lr2, best_rf2]:
    pipe = exp2_pipes[name]
    yp = pipe.predict(X_test)
    ypr = pipe.predict_proba(X_test)[:, 1]
    s = scores_binary(y_bin_test, yp, ypr)
    s["model"] = name
    exp2_test_rows.append(s)
    print(f"\n===== {name} — TEST report (Exp2 binary) =====")
    print(classification_report(y_bin_test, yp, target_names=["Normal", "Attack"], zero_division=0))
    plot_cm(y_bin_test, yp, f"Exp2 TEST CM — {name}")
    # ROC curve
    fpr, tpr, _ = roc_curve(y_bin_test, ypr)
    plt.figure(figsize=(5, 4)); plt.plot(fpr, tpr, label=f"AUC={s['roc_auc']:.3f}"); plt.plot([0,1],[0,1],"--")
    plt.xlabel("FPR"); plt.ylabel("TPR"); plt.title(f"Exp2 ROC — {name}"); plt.legend(); plt.tight_layout(); plt.show()

exp2_test = pd.DataFrame(exp2_test_rows).set_index("model")
print(exp2_test.round(4).to_string())

# Exp1 vs Exp2 comparison (accuracy + weighted/binary F1)
print("\n--- Exp1 (multi, weighted-F1) vs Exp2 (binary F1) on TEST ---")
comp = pd.DataFrame({
    "Exp1_multi_acc": exp1_test["accuracy"], "Exp1_multi_F1w": exp1_test["f1_weighted"],
    "Exp2_bin_acc": exp2_test["accuracy"], "Exp2_bin_F1": exp2_test["f1"]})
print(comp.round(4).to_string())


**Exp2 vs Exp1 (expected):** binary Accuracy/F1 markedly higher (fewer decision boundaries, ~56k+ samples per class vs 130 Worms). State this comparison explicitly in the report.


## 7. Experiment 3 — Binary subset `Normal vs Worms` + class imbalance (intelligent handling)
**Why this pair:** most extreme ratio in the dataset (56000 : 130 ≈ 430:1 in train) → imbalance methods show the clearest trade-off.
**Intelligent protocol:**
1. `baseline` — no handling (reference: already strong at Worms recall 0.795 test / 0.923 val because Worms are separable and 104 train Worms suffice for a deep RF).
2. `class_weight='balanced'` — guideline-compliant, zero extra dependency (neutral here: identical to baseline, free safety net).
3. `SMOTE(k=3)` — inside `imblearn.Pipeline` so oversampling applies **only to training folds** (no leakage). `k_neighbors=3` because only 104 Worms in train (k=5 would bridge outliers). Boosts recall at a precision cost.
4. `LogReg balanced` — linear reference (counter-example: high recall but collapsed precision).
Winner selected on **minority (Worms) Recall / F1**, not accuracy (accuracy is ~99.9% for all RF variants).


In [ ]:
# --- Exp3: build Normal-vs-Worms subset AFTER split logic (filter each partition) ---
def subset_nw(X, y_multi):
    m = y_multi.isin(["Normal", "Worms"])
    Xs = X.loc[m].copy()
    ys = (y_multi.loc[m] == "Worms").astype(int)   # 0=Normal, 1=Worms(minority)
    return Xs, ys

Xtr_nw, ytr_nw = subset_nw(X_tr, y_multi_tr)
Xva_nw, yva_nw = subset_nw(X_val, y_multi_val)
Xte_nw = test_df[FEATURES][test_df["attack_cat"].isin(["Normal", "Worms"])]
yte_nw = (test_df.loc[test_df["attack_cat"].isin(["Normal", "Worms"]), "attack_cat"] == "Worms").astype(int)
print("Subset sizes — train:", Xtr_nw.shape, ytr_nw.value_counts().to_dict(),
      "| val:", Xva_nw.shape, yva_nw.value_counts().to_dict(),
      "| test:", Xte_nw.shape, yte_nw.value_counts().to_dict())
print(f"Imbalance ratio (train Normal:Worms) ≈ {sum(ytr_nw==0)}:{sum(ytr_nw==1)} = {sum(ytr_nw==0)/max(sum(ytr_nw==1),1):.0f}:1")


In [ ]:
# --- Exp3: four strategies, same RF backbone (3x RF + one LogReg reference) ---
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

rf_base = {"n_estimators": 200, "max_depth": 20, "n_jobs": -1, "random_state": RANDOM_STATE}

def prep_clone():
    return clone(preprocessor)

strategies = {
    "RF baseline (no handling)":      Pipeline([("prep", prep_clone()), ("clf", RandomForestClassifier(class_weight=None, **rf_base))]),
    "RF class_weight=balanced":        Pipeline([("prep", prep_clone()), ("clf", RandomForestClassifier(class_weight="balanced", **rf_base))]),
    "RF SMOTE(k=3)+RF":                ImbPipeline([("prep", prep_clone()), ("smote", SMOTE(random_state=RANDOM_STATE, k_neighbors=3)), ("clf", RandomForestClassifier(**rf_base))]),
    "LogReg class_weight=balanced":    Pipeline([("prep", prep_clone()), ("clf", LogisticRegression(C=1.0, max_iter=500, class_weight="balanced", random_state=RANDOM_STATE))]),
}

exp3_rows = []
for name, pipe in strategies.items():
    pipe.fit(Xtr_nw, ytr_nw)   # SMOTE/scaler fitted ONLY on train subset
    for split, (Xs, ys) in {"val": (Xva_nw, yva_nw), "test": (Xte_nw, yte_nw)}.items():
        yp = pipe.predict(Xs)
        ypr = pipe.predict_proba(Xs)[:, 1]
        s = scores_binary(ys, yp, ypr)
        s.update({"strategy": name, "split": split,
                  "minority_recall": recall_score(ys, yp, zero_division=0),
                  "minority_precision": precision_score(ys, yp, zero_division=0)})
        exp3_rows.append(s)
    yp_t = pipe.predict(Xte_nw)
    print(f"\n===== {name} — TEST report (Worms=1) =====")
    print(classification_report(yte_nw, yp_t, target_names=["Normal", "Worms"], zero_division=0))

exp3_all = pd.DataFrame(exp3_rows)
print("\nExp3 VAL vs TEST (accuracy can lie — watch minority recall):")
print(exp3_all.round(4).to_string(index=False))

for name in strategies:
    yp = strategies[name].predict(Xte_nw)
    plot_cm(yte_nw, yp, f"Exp3 TEST CM — {name}")


In [ ]:
# Exp3 ROC overlay — visual proof of intelligent handling
plt.figure(figsize=(6, 5))
for name, pipe in strategies.items():
    ypr = pipe.predict_proba(Xte_nw)[:, 1]
    fpr, tpr, _ = roc_curve(yte_nw, ypr)
    auc = roc_auc_score(yte_nw, ypr)
    plt.plot(fpr, tpr, label=f"{name} (AUC={auc:.3f})")
plt.plot([0, 1], [0, 1], "--", color="gray")
plt.xlabel("FPR"); plt.ylabel("TPR (Worms recall)"); plt.title("Exp3 ROC — Normal vs Worms (TEST)")
plt.legend(fontsize=8); plt.tight_layout(); plt.show()

pivot = exp3_all.pivot(index="strategy", columns="split", values=["accuracy", "f1", "minority_recall"])
print(pivot.round(4).to_string())


**Exp3 reading guide (actual results, TEST on 37000 Normal / 44 Worms):**
- Baseline RF: accuracy 0.9996 hides 9/44 missed Worms (recall 0.795, precision 0.897, F1 0.843). Do not trust accuracy alone — but recall is NOT 0 here: Worms are separable and 104 train Worms are enough for a depth-20 RF (val recall 0.923 = 24/26).
- `class_weight='balanced'`: neutral on this subset — identical to baseline on val (0.923/0.923) and test (0.795/0.897). No gain, no harm; keep as free default.
- SMOTE(k=3)+RF: recall 0.796 → 0.841 (+2 Worms: 35 → 37/44) at precision 0.897 → 0.725 (+10 FP: 4 → 14), F1 0.843 → 0.779. Val confirms (recall 0.923 → 0.962 = 24/26 → 25/26). Heavier + must stay inside pipeline (leakage if applied before split).
- LogReg balanced: recall 0.864 (38/44) but precision 0.029 (~1264 FP) → unusable. Proves a linear model cannot handle this with re-weighting alone.
- Verdict: default to `class_weight='balanced'`; add SMOTE when val minority recall is still below target AND you can afford extra false positives — exactly the Worms case.


## 8. Synthesis — comparative tables + saved artifacts


In [ ]:
print("===== FINAL COMPARATIVE TABLE (TEST) =====")
print("\n-- Exp1 multi-class --"); print(exp1_test.round(4).to_string())
print("\n-- Exp2 binary --"); print(exp2_test.round(4).to_string())
print("\n-- Exp3 Normal-vs-Worms (test) --")
print(exp3_all[exp3_all.split == "test"].set_index("strategy").round(4).to_string())

# Bar comparison: best RF per experiment (F1)
import pandas as pd
summary = pd.DataFrame({
    "experiment": ["Exp1 multi (F1-weighted)", "Exp2 binary (F1)", "Exp3 N-vs-W (F1, balanced RF)"],
    "best_RF_F1": [float(exp1_test.loc[best_rf_name, "f1_weighted"]),
                   float(exp2_test.loc[best_rf2, "f1"]),
                   float(exp3_all[(exp3_all.strategy == "RF class_weight=balanced") & (exp3_all.split == "test")]["f1"].iloc[0])]
})
plt.figure(figsize=(8, 4))
sns.barplot(x="experiment", y="best_RF_F1", data=summary, palette="mako")
plt.ylim(0, 1.02); plt.title("Best RF F1 per experiment (TEST)"); plt.xticks(rotation=8)
for i, v in enumerate(summary["best_RF_F1"]):
    plt.text(i, v + 0.02, f"{v:.3f}", ha="center", fontweight="bold")
plt.tight_layout(); plt.show()

# Save tables for the report
exp1_test.to_csv("results_exp1_test.csv")
exp2_test.to_csv("results_exp2_test.csv")
exp3_all.to_csv("results_exp3_all.csv", index=False)
print("Saved: results_exp1_test.csv, results_exp2_test.csv, results_exp3_all.csv")


## 9. Conclusion (to expand in `report.md`)
1. Pipeline + stratified split prevented leakage; scalers/encoders never saw val/test.
2. RF > LogReg everywhere; LogReg underfits multi-class, RF overfits slightly when unbounded.
3. Binary ≫ multi-class (expected): collapsing 9 attacks into one class removes fine-grained confusion.
4. On Normal-vs-Worms (44 test Worms), accuracy lies (0.9996 baseline hides 20% missed = 9/44); `balanced` was neutral (identical to baseline), SMOTE boosted recall 0.796 → 0.841 at precision 0.897 → 0.725; LogReg collapsed (precision 0.029, ~1264 FP).
